# TA-07: Q-Learning untuk Kurir Paket Cerdas (Grid 4×4)
**Nama:** Soni Moch Leviansyah | **NIM:** 301240030

Notebook ini berisi environment (Python murni), algoritma Q-Learning, kurva pembelajaran, visualisasi policy, dan eksperimen hyperparameter γ.

## 1. Environment

In [ ]:
"""Environment Kurir Paket Cerdas (grid 4x4) - Python murni, tanpa Gymnasium."""
import numpy as np

class KurirEnv:
    ATAS, BAWAH, KIRI, KANAN, AMBIL, ANTAR = range(6)
    NAMA_AKSI = ["Atas", "Bawah", "Kiri", "Kanan", "Ambil", "Antar"]
    SIMBOL = {0: "↑", 1: "↓", 2: "←", 3: "→", 4: "A", 5: "T"}

    def __init__(self):
        self.n_baris = self.n_kolom = 4
        self.start, self.titik_p, self.titik_d = (0, 0), (0, 3), (3, 0)
        self.banjir = {(1, 1), (2, 2)}
        self.n_posisi = self.n_baris * self.n_kolom
        self.n_states = self.n_posisi * 2          # 16 x 2 = 32
        self.n_actions = 6
        self.reset()

    def encode(self, pos, paket):
        return (pos[0] * self.n_kolom + pos[1]) * 2 + paket

    def decode(self, s):
        p, paket = divmod(s, 2)
        return divmod(p, self.n_kolom), paket

    def reset(self):
        self.pos, self.paket = self.start, 0
        return self.encode(self.pos, self.paket)

    def step_from(self, state, action):
        """Fungsi transisi murni: (state, action) -> (new_state, reward, done)."""
        pos, paket = self.decode(state)
        r, c = pos
        reward, done = -1, False
        if action <= self.KANAN:
            dr, dc = [(-1, 0), (1, 0), (0, -1), (0, 1)][action]
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.n_baris and 0 <= nc < self.n_kolom:
                pos = (nr, nc)
                if pos in self.banjir:
                    reward, done = -20, True
            else:
                reward = -5                      # menabrak tepi peta
        elif action == self.AMBIL:
            if pos == self.titik_p and paket == 0:
                paket, reward = 1, 10
            else:
                reward = -10
        else:                                    # ANTAR
            if pos == self.titik_d and paket == 1:
                reward, done = 30, True
            else:
                reward = -10
        return self.encode(pos, paket), reward, done

    def step(self, action):
        s = self.encode(self.pos, self.paket)
        ns, reward, done = self.step_from(s, action)
        self.pos, self.paket = self.decode(ns)
        return ns, reward, done


## 2. Algoritma Q-Learning (ε-greedy)

In [ ]:
import numpy as np

def train(alpha=0.8, gamma=0.9, episodes=2000, max_steps=100, eps_start=1.0, eps_min=0.01,
          decay=0.005, seed=42):
    rng = np.random.default_rng(seed)
    env = KurirEnv()
    Q = np.zeros((env.n_states, env.n_actions))
    rewards, steps_log, eps_log = [], [], []
    for ep in range(episodes):
        eps = eps_min + (eps_start - eps_min) * np.exp(-decay * ep)
        s, total, n = env.reset(), 0, 0
        for _ in range(max_steps):
            a = rng.integers(env.n_actions) if rng.random() < eps else int(np.argmax(Q[s]))
            ns, r, done = env.step(a)
            target = r if done else r + gamma * np.max(Q[ns])      # state terminal: tanpa bootstrap
            Q[s, a] = (1 - alpha) * Q[s, a] + alpha * target
            s, total, n = ns, total + r, n + 1
            if done:
                break
        rewards.append(total); steps_log.append(n); eps_log.append(eps)
    return env, Q, np.array(rewards), np.array(steps_log), np.array(eps_log)

def greedy_rollout(env, Q, max_steps=50):
    s, total, path = env.reset(), 0, [env.start]
    acts = []
    for _ in range(max_steps):
        a = int(np.argmax(Q[s])); acts.append(a)
        s, r, done = env.step(a); total += r; path.append(env.pos)
        if done: break
    return total, acts, path, done

def moving_avg(x, w=100):
    import pandas as pd
    return pd.Series(x).rolling(w, min_periods=1).mean().values



## 3. Pelatihan dasar dan kurva pembelajaran
Hyperparameter: α=0.8, γ=0.9, ε: 1.0 → 0.01 (decay 0.005), 2000 episode.

In [ ]:
import matplotlib.pyplot as plt
env, Q, R, S, E = train(alpha=0.8, gamma=0.9, episodes=2000, seed=42)
ma = moving_avg(R, 100)
conv = int(np.argmax(ma >= 0.95 * ma[-100:].mean()))
print('Episode konvergensi (MA mencapai 95% plateau):', conv)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(R, color='lightgray', lw=.5); ax[0].plot(ma, color='green'); ax[0].axvline(conv, color='red', ls='--')
ax[0].set_title('Kurva Pembelajaran'); ax[0].set_xlabel('Episode'); ax[0].set_ylabel('Reward'); ax[0].grid(True)
ax[1].plot(E, color='firebrick'); ax[1].set_title('Peluruhan epsilon'); ax[1].set_xlabel('Episode'); ax[1].grid(True)
plt.show()

## 4. Policy optimal (argmax Q)

In [ ]:
total, acts, path, done = greedy_rollout(env, Q)
print('Reward greedy:', total, '| langkah:', len(acts), '| selesai:', done)
print([env.NAMA_AKSI[a] for a in acts])
fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
for paket in (0, 1):
    grid = np.array([[Q[env.encode((r, c), paket)].max() for c in range(4)] for r in range(4)])
    ax[paket].imshow(grid, cmap='YlGn')
    for r in range(4):
        for c in range(4):
            lab = 'X' if (r, c) in env.banjir else env.SIMBOL[int(np.argmax(Q[env.encode((r, c), paket)]))]
            ax[paket].text(c, r, lab, ha='center', va='center', fontsize=16, fontweight='bold')
    ax[paket].set_title('Paket belum diambil' if paket == 0 else 'Paket sedang dibawa'); ax[paket].set_xticks([]); ax[paket].set_yticks([])
plt.show()

## 5. Eksperimen hyperparameter: γ = 0.9 vs γ = 0.1 (10 seed)

In [ ]:
for g in (0.9, 0.1):
    hasil, margin = [], []
    for sd in range(10):
        e, q, r, s, _ = train(gamma=g, seed=sd)
        t, a, p, d = greedy_rollout(e, q); hasil.append(t)
        qs = np.sort(q[e.encode(e.start, 0)])[::-1]; margin.append(qs[0] - qs[1])
    print(f'gamma={g}: reward greedy rata-rata={np.mean(hasil):.1f}, selisih Q terbaik-kedua di state awal={np.mean(margin):.3f}')